# Trabajo Práctico — Series Temporales

## Ozono, Clima y Radiación Solar en la Cuenca de Los Ángeles (California)

**Maestría en Ciencia de Datos / Análisis de Series Temporales**

---

**Nota sobre el origen de este proyecto:** este notebook reemplaza a un proyecto anterior basado en demanda eléctrica, generación y temperatura del AMBA (Argentina). Se decidió migrar a esta combinación porque las fuentes horarias reales de CAMMESA no permiten superar ampliamente los 100.000 registros por serie sin recurrir a trucos de conteo (sumar provincias/tecnologías como si fueran filas independientes), mientras que esta combinación en California sí permite superarlo ampliamente con una sola serie limpia por variable. El contenido del proyecto anterior queda conservado, sin borrar, en `_archivo_TP1_AMBA/`.

## Objetivo del notebook

Este notebook **no realiza análisis estadístico**. Su función es:

- crear la estructura de carpetas del proyecto,
- descargar las tres series horarias desde sus fuentes oficiales,
- dejarlas guardadas sin modificar en `data/raw/`,
- y validar que la descarga fue completa,

para que los siguientes notebooks (`02_limpieza.ipynb`, `03_eda.ipynb`, etc.) trabajen sobre datos ya adquiridos.

## Las tres series y por qué correlacionan entre sí

| Serie | Fuente | Qué mide | Frecuencia nativa |
|---|---|---|---|
| Ozono troposférico (O₃) | EPA AQS / AirData | Calidad del aire | Horaria |
| Temperatura y clima | NOAA ISD (Global Hourly) | Condiciones meteorológicas | Horaria |
| Radiación solar | NSRDB (NLR, ex-NREL) | Irradiancia solar | 30 minutos |

No son tres series elegidas al azar: el **ozono troposférico se forma por fotoquímica solar** (NOx + compuestos orgánicos volátiles + luz solar → ozono), un mecanismo bien documentado en química atmosférica. Por eso la radiación solar y la temperatura explican gran parte de la variación horaria y estacional del ozono — la misma lógica causal que usábamos antes con temperatura → demanda eléctrica.

## Ubicación elegida: Los Ángeles, California

Para que las tres series sean comparables entre sí (mismo punto geográfico, mismo huso horario, misma cuenca atmosférica), se fija un único punto de referencia:

- **Clima y radiación solar**: coordenadas de LAX (33.9425, -118.4081).
- **Ozono**: estación de monitoreo de EPA AQS *Los Angeles-North Main Street* (código de sitio `06-037-1103`), a pocos kilómetros de LAX, dentro de la misma cuenca del aire (South Coast Air Basin) — es una de las estaciones de ozono con historial más largo de California.

## Períodos de cada fuente (verificado, no estimado)

- **EPA AQS**: archivos horarios públicos desde **1990**.
- **NOAA ISD (LAX)**: cobertura horaria desde **1944**.
- **NSRDB GOES Aggregated**: desde **1998** — esta es la fuente que acota el período común del proyecto.

Por lo tanto, el rango efectivo del proyecto es **1998 – año actual - 1** (se excluye el año en curso porque los tres organismos publican sus archivos anuales con varios meses de demora). Con ese rango, cada serie por sí sola ya supera ampliamente los 100.000 registros horarios, sin necesidad de sumar estaciones ni provincias.

## Licencias y uso de datos (verificado)

Las tres fuentes son de dominio público / uso libre para investigación académica, sin necesidad de permiso:

- **EPA AQS**: dominio público, sin restricciones, no requiere pedir permiso.
- **NOAA/NCEI**: datos del gobierno federal de EE.UU., política de dominio público (CC0), se recomienda citar como buena práctica.
- **NSRDB (NLR)**: pública y gratuita; sólo requiere una API key propia (autoservicio, ver sección 4 de este notebook). El único límite real: no usar el nombre de Caltrans/agencias para publicidad — no aplica a un TP académico.

## 1. Configuración inicial y estructura de carpetas

Se detecta la raíz del proyecto y se crea (si no existe) la estructura completa: `data/raw/{epa_aqs,noaa_isd,nsrdb}`, `data/interim`, `data/processed`, `notebooks`, `src`, `outputs/{figuras,tablas,modelos,reportes}`, `docs`, `logs`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Directorio raíz del proyecto detectado: {PROJECT_ROOT}")

Directorio raíz del proyecto detectado: C:\Series_AMBA


In [2]:
from utils import crear_estructura_proyecto, info_entorno, configurar_logging

creadas, existentes = crear_estructura_proyecto(PROJECT_ROOT)

print(f"Carpetas creadas ahora ({len(creadas)}):")
for c in creadas:
    print(f"  + {c}")
print(f"\nCarpetas que ya existían ({len(existentes)}):")
for c in existentes:
    print(f"  = {c}")

Carpetas creadas ahora (0):

Carpetas que ya existían (13):
  = C:\Series_AMBA\data\raw\epa_aqs
  = C:\Series_AMBA\data\raw\noaa_isd
  = C:\Series_AMBA\data\raw\nsrdb
  = C:\Series_AMBA\data\interim
  = C:\Series_AMBA\data\processed
  = C:\Series_AMBA\notebooks
  = C:\Series_AMBA\src
  = C:\Series_AMBA\outputs\figuras
  = C:\Series_AMBA\outputs\tablas
  = C:\Series_AMBA\outputs\modelos
  = C:\Series_AMBA\outputs\reportes
  = C:\Series_AMBA\docs
  = C:\Series_AMBA\logs


In [3]:
print("Proyecto inicializado — información del entorno")
print("=" * 55)
for clave, valor in info_entorno().items():
    print(f"{clave:.<35}{valor}")

Proyecto inicializado — información del entorno
sistema_operativo..................Windows 11
version_so_detallada...............10.0.26200
version_python.....................3.12.13
arquitectura.......................AMD64
memoria_total_gb...................15.69
memoria_disponible_gb..............0.94


In [4]:
logger = configurar_logging(PROJECT_ROOT / "logs" / "descarga_datos.log")
logger.info("Notebook 01_descarga_datos (California) iniciado")
print(f"Log de esta ejecución: {PROJECT_ROOT / 'logs' / 'descarga_datos.log'}")

2026-08-07 00:09:23 | INFO     | Notebook 01_descarga_datos (California) iniciado


Log de esta ejecución: C:\Series_AMBA\logs\descarga_datos.log


## 2. Rango de años a descargar

Se fija el rango común a las tres fuentes: **1998 hasta el año actual - 1** (el año en curso todavía no tiene archivo anual publicado en ninguna de las tres fuentes). Cada función de descarga es resumible: si se interrumpe y se vuelve a ejecutar, los años ya guardados en disco no se vuelven a pedir.

In [5]:
from datetime import date

ANIO_INICIO = 1998  # límite real de NSRDB (GOES Aggregated)
ANIO_FIN = date.today().year - 1

print(f"Rango a descargar: {ANIO_INICIO}-{ANIO_FIN} ({ANIO_FIN - ANIO_INICIO + 1} años)")

Rango a descargar: 1998-2025 (28 años)


## 3. EPA AQS / AirData — ozono horario

**¿Qué es?** El sistema oficial de calidad del aire de la EPA (Agencia de Protección Ambiental de EE.UU.). Publica un archivo `.zip` por año y por contaminante con TODAS las estaciones del país (`hourly_44201_{año}.zip` para ozono).

**¿Por qué ozono?** Es el contaminante cuya formación depende más directamente de la radiación solar (fotoquímica), lo que da la correlación buscada con la tercera serie del proyecto.

Se descarga el archivo nacional completo tal cual (sin filtrar la estación de Los Ángeles): el recorte a la estación `06-037-1103` se hace en `02_limpieza.ipynb`, para no transformar el dato crudo en esta etapa.

In [6]:
from descarga import descargar_epa_aqs_rango, EPA_AQS_PARAMETROS

destino_epa = PROJECT_ROOT / "data/raw/epa_aqs"

resumen_epa = descargar_epa_aqs_rango(
    anio_inicio=ANIO_INICIO,
    anio_fin=ANIO_FIN,
    destino_dir=destino_epa,
    parametro_codigo=EPA_AQS_PARAMETROS["OZONO"],
    logger=logger,
    pausa_seg=1.0,
    reintentos=3,
    espera_reintento_seg=5.0,
)

print(f"Años descargados ahora      : {resumen_epa.anios_ok}")
print(f"Años ya existentes en disco : {resumen_epa.anios_ya_existentes}")
print(f"Años sin archivo publicado  : {resumen_epa.anios_sin_datos}")
print(f"Años con error de descarga  : {resumen_epa.anios_con_error} (ya reintentados 3 veces cada uno)")
if resumen_epa.errores:
    print(f"Años que fallaron incluso después de los reintentos: {resumen_epa.errores}")

2026-08-07 00:11:46 | INFO     | EPA AQS 2011: descargado


2026-08-07 00:13:56 | INFO     | EPA AQS 2012: descargado


2026-08-07 00:16:15 | INFO     | EPA AQS 2013: descargado


2026-08-07 00:18:32 | INFO     | EPA AQS 2014: descargado


2026-08-07 00:20:51 | INFO     | EPA AQS 2015: descargado


...

2026-08-07 00:35:54 | INFO     | EPA AQS 2022: descargado


2026-08-07 00:38:05 | INFO     | EPA AQS 2023: descargado


2026-08-07 00:40:36 | INFO     | EPA AQS 2024: descargado


2026-08-07 00:42:07 | INFO     | EPA AQS 2025: descargado


Años descargados ahora      : 15
Años ya existentes en disco : 13
Años sin archivo publicado  : 0
Años con error de descarga  : 0 (ya reintentados 3 veces cada uno)


## 4. NOAA ISD (Global Hourly) — clima horario en LAX

**¿Qué es?** La Base de Datos de Superficie Integrada de NOAA/NCEI: observaciones horarias y subhorarias de ~35.500 estaciones meteorológicas en todo el mundo. Se usa la estación de **LAX** (ID combinado USAF+WBAN = `72295023174`), con cobertura horaria confiable desde 1944.

**¿Qué representa la serie?** Temperatura, punto de rocío, presión, viento y otras variables de superficie, en formato de texto delimitado por comas, un archivo por año y por estación.

In [7]:
from descarga import descargar_noaa_isd_rango, NOAA_ISD_ESTACION_LAX

destino_noaa = PROJECT_ROOT / "data/raw/noaa_isd"

resumen_noaa = descargar_noaa_isd_rango(
    anio_inicio=ANIO_INICIO,
    anio_fin=ANIO_FIN,
    destino_dir=destino_noaa,
    estacion_id=NOAA_ISD_ESTACION_LAX,
    logger=logger,
    pausa_seg=1.0,
    reintentos=3,
    espera_reintento_seg=5.0,
)

print(f"Años descargados ahora      : {resumen_noaa.anios_ok}")
print(f"Años ya existentes en disco : {resumen_noaa.anios_ya_existentes}")
print(f"Años sin archivo publicado  : {resumen_noaa.anios_sin_datos}")
print(f"Años con error de descarga  : {resumen_noaa.anios_con_error} (ya reintentados 3 veces cada uno)")
if resumen_noaa.errores:
    print(f"Años que fallaron incluso después de los reintentos: {resumen_noaa.errores}")

2026-08-07 00:42:16 | INFO     | NOAA ISD 1998: descargado


2026-08-07 00:42:20 | INFO     | NOAA ISD 1999: descargado


2026-08-07 00:42:23 | INFO     | NOAA ISD 2000: descargado


2026-08-07 00:42:26 | INFO     | NOAA ISD 2001: descargado


2026-08-07 00:42:30 | INFO     | NOAA ISD 2002: descargado


...

2026-08-07 00:43:34 | INFO     | NOAA ISD 2022: descargado


2026-08-07 00:43:38 | INFO     | NOAA ISD 2023: descargado


2026-08-07 00:43:41 | INFO     | NOAA ISD 2024: descargado


2026-08-07 00:43:44 | INFO     | NOAA ISD 2025: descargado


Años descargados ahora      : 28
Años ya existentes en disco : 0
Años sin archivo publicado  : 0
Años con error de descarga  : 0 (ya reintentados 3 veces cada uno)


## 5. NSRDB (NLR, ex-NREL) — radiación solar

**Nota sobre el nombre:** el organismo que mantiene esta base (antes NREL, "National Renewable Energy Laboratory") se renombró a **NLR** ("National Laboratory of the Rockies"). El dominio viejo (`developer.nrel.gov`) ya no resuelve; el actual es `developer.nlr.gov`. Se verificó en vivo antes de escribir este notebook.

**Antes de correr la celda de descarga, cada integrante del grupo tiene que sacar su propia API key gratuita:**

1. Ir a **https://developer.nlr.gov/signup/**.
2. Completar el formulario (nombre, email — autoservicio, sin aprobación manual, a diferencia de Caltrans PeMS).
3. La API key llega por email al instante.
4. Completar `API_KEY_NSRDB` y `EMAIL_NSRDB` en la celda siguiente (o definirlas como variables de entorno `NSRDB_API_KEY` / `NSRDB_EMAIL` antes de abrir Jupyter, para no dejar la key escrita en el notebook).

Si no se completa la API key, la celda de descarga lo detecta y no manda ningún pedido al servidor (evita gastar cupo de la API en vano).

In [8]:
import os

# Preferí completar esto vía variables de entorno (NSRDB_API_KEY / NSRDB_EMAIL)
# antes que pegar la key directamente en el notebook.
API_KEY_NSRDB = os.environ.get("NSRDB_API_KEY", "lMoZr1PvDIegNvTfMP1UFk5z6IhHZsm06ceNUv9Y")
EMAIL_NSRDB = os.environ.get("NSRDB_EMAIL", "lggastaldo@mail.austral.edu.ar")

if not API_KEY_NSRDB or not EMAIL_NSRDB:
    print("Falta API_KEY_NSRDB y/o EMAIL_NSRDB. Sacar una key gratis en https://developer.nlr.gov/signup/")
    print("y completarla acá (o como variable de entorno) antes de reintentar esta sección.")
else:
    print("API key de NSRDB detectada, lista para descargar.")

API key de NSRDB detectada, lista para descargar.


In [9]:
from descarga import descargar_nsrdb_rango

destino_nsrdb = PROJECT_ROOT / "data/raw/nsrdb"

resumen_nsrdb = descargar_nsrdb_rango(
    anio_inicio=ANIO_INICIO,
    anio_fin=ANIO_FIN,
    api_key=API_KEY_NSRDB,
    email=EMAIL_NSRDB,
    destino_dir=destino_nsrdb,
    logger=logger,
    pausa_seg=2.0,
    reintentos=3,
    espera_reintento_seg=5.0,
)

print(f"Años descargados ahora      : {resumen_nsrdb.anios_ok}")
print(f"Años ya existentes en disco : {resumen_nsrdb.anios_ya_existentes}")
print(f"Años con error de descarga  : {resumen_nsrdb.anios_con_error} (ya reintentados 3 veces cada uno)")
if resumen_nsrdb.errores:
    print(f"Años que fallaron incluso después de los reintentos: {resumen_nsrdb.errores}")

2026-08-07 00:44:12 | INFO     | NSRDB 1998: descargado


2026-08-07 00:44:27 | INFO     | NSRDB 1999: descargado


2026-08-07 00:44:42 | INFO     | NSRDB 2000: descargado


2026-08-07 00:44:56 | INFO     | NSRDB 2001: descargado


2026-08-07 00:45:11 | INFO     | NSRDB 2002: descargado


...

2026-08-07 00:50:00 | INFO     | NSRDB 2022: descargado


2026-08-07 00:50:14 | INFO     | NSRDB 2023: descargado


2026-08-07 00:50:28 | INFO     | NSRDB 2024: descargado


2026-08-07 00:50:42 | INFO     | NSRDB 2025: descargado


Años descargados ahora      : 28
Años ya existentes en disco : 0
Años con error de descarga  : 0 (ya reintentados 3 veces cada uno)


## 6. Resumen de calidad de los datos descargados

Cantidad de archivos anuales presentes en disco por fuente, antes de pasar a `02_limpieza.ipynb`. Este notebook no parsea el contenido de cada archivo (eso es tarea de la limpieza); sólo valida que los años esperados están efectivamente descargados.

In [10]:
def contar_anios_en_disco(destino_dir, sufijo):
    destino_dir = Path(destino_dir)
    if not destino_dir.exists():
        return []
    return sorted(p.name for p in destino_dir.glob(f"*{sufijo}"))

archivos_epa = contar_anios_en_disco(destino_epa, ".zip")
archivos_noaa = contar_anios_en_disco(destino_noaa, ".csv")
archivos_nsrdb = contar_anios_en_disco(destino_nsrdb, ".csv")

anios_esperados = ANIO_FIN - ANIO_INICIO + 1

print(f"Rango esperado: {ANIO_INICIO}-{ANIO_FIN} ({anios_esperados} años)\n")
print(f"EPA AQS (ozono)  : {len(archivos_epa)}/{anios_esperados} archivos anuales en disco")
print(f"NOAA ISD (LAX)   : {len(archivos_noaa)}/{anios_esperados} archivos anuales en disco")
print(f"NSRDB (solar)    : {len(archivos_nsrdb)}/{anios_esperados} archivos anuales en disco")

if len(archivos_nsrdb) == 0:
    print("\nNSRDB en 0: falta completar la API key de la sección 5 antes de tener esta serie.")

Rango esperado: 1998-2025 (28 años)

EPA AQS (ozono)  : 28/28 archivos anuales en disco
NOAA ISD (LAX)   : 28/28 archivos anuales en disco
NSRDB (solar)    : 28/28 archivos anuales en disco


## 7. Cierre

- ✅ Carpetas creadas
- ✅ EPA AQS (ozono horario) — descarga automática, sin autenticación
- ✅ NOAA ISD (clima horario, LAX) — descarga automática, sin autenticación
- ⏳ NSRDB (radiación solar) — automática, pero requiere que cada integrante complete su propia API key gratuita de `developer.nlr.gov`

El contenido del proyecto anterior (AMBA: CAMMESA + SMN) queda conservado sin borrar en `_archivo_TP1_AMBA/`, por si se necesita retomarlo.

**Proyecto listo para continuar con `02_limpieza.ipynb`** (unir los archivos anuales de cada fuente en una única serie horaria por variable, recortar a la estación de Los Ángeles en el caso de EPA AQS, y homogeneizar el índice temporal de las tres).